In [ ]:
import json
import os
with open("/public/zhangzhiling/code/Robust-Wide/filtered_datasets/max1000/timbrooks___instructpix2pix-clip-filtered/magicbrush-jul7/metadata.json", 'r', encoding='utf-8') as f:
    metadata = json.load(f)

print(metadata.keys())
prompts = metadata["edit_prompt"]
print(metadata['filter_stats'])
images_dir = "/public/zhangzhiling/code/Robust-Wide/filtered_datasets/max1000/timbrooks___instructpix2pix-clip-filtered/magicbrush-jul7/images"
image_files = [f for f in os.listdir(images_dir) if f.endswith('.png')]
image_files.sort()
image_files[:3]
dataset_dict = {
    "image": [os.path.join(images_dir, img_file) for img_file in image_files],
    "edit_prompt": prompts
}
dataset_dict["image"][:3]
dataset_dict["edit_prompt"][:3]

: 

In [ ]:
from datasets import Features, Image, Value, Dataset

features = Features({
    "image": Image(),
    "edit_prompt": Value("string")
})
filtered_dataset = Dataset.from_dict(dataset_dict, features=features)
    

In [ ]:
train_size=20000
total_samples = len(filtered_dataset)
if total_samples < train_size:
    print(f"筛选数据集样本数量({total_samples})小于请求的训练集大小({train_size})，使用所有样本")
    train_dataset = filtered_dataset

else:
    train_dataset = filtered_dataset.select(range(train_size))

In [ ]:
import torch
import numpy as np
from torchvision import transforms
from PIL import Image

def convert_to_np(image, resolution):
    image = np.array(image)
    image = image.astype(np.float32) / 255.0
    image = image.transpose(2, 0, 1)  # 转换为 (C, H, W) 格式
def preprocess_train(examples, image_size):
    train_transforms = transforms.Compose(
        [
            transforms.Resize(int(image_size * 1.1), interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.RandomCrop(image_size),
            transforms.RandomHorizontalFlip(),
        ]
    )
    
    # 打印 examples 的 keys
    print("Examples keys:", examples.keys())
    
    # 根据键选择图像
    image_keys = ["original_image", "source_image", "source_img", "image"]
    images = None
    
    for key in image_keys:
        if key in examples:
            images = np.concatenate(
                [convert_to_np(image, image_size) for image in examples[key]]
            )
            break  # 找到第一个匹配的键后退出循环
    
    if images is None:
        raise KeyError("None of the expected image keys found in examples.")
    
    images = torch.tensor(images)
    images = 2 * (images / 255) - 1
    images = train_transforms(images)
    
    # 根据键选择 prompt
    prompt_keys = ["edit_prompt", "instruction"]
    prompt = None

    for key in prompt_keys:
        if key in examples:
            prompt = list(examples[key])  # 转换为列表
            break  # 找到第一个匹配的键后退出循环

    if prompt is None:
        raise KeyError("None of the expected prompt keys found in examples.")

    examples["image"] = images.reshape(-1, 3, image_size, image_size)
    examples["prompt"] = prompt
    return examples

In [ ]:
from functools import partial

train_dataset = train_dataset.with_transform(partial(preprocess_train, image_size=512))


In [ ]:
from torch.utils.data import DataLoader
import sys
sys.path.append('/public/zhangzhiling/code/Robust-Wide')
from dataset import get_hugging_dataset, get_filtered_dataset, collate_fn 
train_dataloader = DataLoader(
    train_dataset,
    batch_size=1,
    drop_last=True,
    # shuffle=True,
    shuffle=False,
    collate_fn=collate_fn,
)

In [ ]:
from IPython.display import display
# display(train_dataset[0]['image'])
for data in train_dataloader:
    image, prompt = data["image"], data["prompt"]
    print(prompt)
    display(image)
    break